# 판례 검색 후 json 파일로 저장하기

In [16]:
# 판례 목록 조회
import os
import requests

def search_prec(query_text: str):
    """
    국가법령정보 API를 통해 법령을 검색하는 함수
    """
    
    # 기본 URL 설정
    base_url = "https://www.law.go.kr/DRF/lawSearch.do"


    params = {
        "OC": "legal_123",
        "target": "prec",
        "type": "json",
        "display": "3000",
        "query": query_text  # 여기에 검색어가 들어갑니다.
    }
    
    try:
        response = requests.get(base_url, params=params, timeout=30)
        
        print(f"--- '{query_text}' 검색 결과 ---")
        print("status =", response.status_code)
        
        # 성공적으로 데이터를 가져왔다면 JSON 형태로 출력하거나 반환
        if response.status_code == 200:
            return response.json()
        else:
            print("오류 발생:", response.text)
            return None
            
    except requests.exceptions.RequestException as e:
        print("요청 중 에러 발생:", e)
        return None



In [35]:
# 판례 본문 조회
import os
import requests

def search_prec_all(data:str):
    """
    국가법령정보 API를 통해 법령을 검색하는 함수
    """
    
    # 기본 URL 설정
    base_url = "http://www.law.go.kr/DRF/lawService.do"


    params = {
        "OC": "legal_123",
        "target": "prec",
        "type": "json",
        "ID" : data
    }
    
    try:
        response = requests.get(base_url, params=params, timeout=30)
        
        print(f"---검색 결과 ---")
        print("status =", response.status_code)
        
        # 성공적으로 데이터를 가져왔다면 JSON 형태로 출력하거나 반환
        if response.status_code == 200:
            return response.json()
        else:
            print("오류 발생:", response.text)
            return None
            
    except requests.exceptions.RequestException as e:
        print("요청 중 에러 발생:", e)
        return None

In [23]:
import os
import requests

def search_prec_save(query_text: str):
    """
    국가법령정보 API를 통해 법령을 검색하는 함수
    """
    prec_id_list_final = [] # 정상적인 판례를 담을 리스트
    prec_id_list = [] # 1차 판례 리스트
    
    # 기본 URL 설정
    base_url = "https://www.law.go.kr/DRF/lawSearch.do"

    params = {
        "OC": "legal_123",
        "target": "prec",
        "type": "json",
        "display": "3000",
        "query": query_text  # 여기에 검색어가 들어갑니다.
    }
    
    try:
        response = requests.get(base_url, params=params, timeout=30)
        
        # 성공적으로 데이터를 가져왔다면 JSON 형태로 출력하거나 반환
        if response.status_code == 200:
            result = response.json()
        else:
            print("오류 발생:", response.text)
            return None
        
        prec_id_list = [item['판례일련번호'] for item in result['PrecSearch']['prec']]

        base_detail_url = "https://www.law.go.kr/DRF/lawService.do"

        # --- 1. 모든 일련번호를 순회하는 반복문 시작 ---
        for id in prec_id_list:
            params = {
                "OC": "legal_123",
                "target": "prec",
                "type": "json",
                "ID": id
            }
            
            try:
                response = requests.get(base_detail_url, params=params)

                if response.status_code != 200:
                    print(f"네트워크 오류 / 상태코드 {response.status_code} - 일련번호 : {id}")
                    continue # 에러 시 다음 id로 넘어가도록 처리
                
                res_json = response.json()
            
                # 1. API 응답 구조 확인 및 PrecService 노드 확보
                prec_service = res_json.get('PrecService', {})
                if not prec_service:
                    print(f"⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: {id}")
                    continue
                    
                # 2. 본문 검색에 필수적인 핵심 필드들이 비어있는지 검증
                판례내용 = str(prec_service.get('판례내용', '')).strip()
                판시사항 = str(prec_service.get('판시사항', '')).strip()
                판결요지 = str(prec_service.get('판결요지', '')).strip()
                
                # 내용, 판시사항, 판결요지가 '모두' 비어있거나 의미 없는 값이라면 본문 조회가 안 된 것으로 판단
                if not 판례내용 and not 판시사항 and not 판결요지:
                    print(f"⚠️ 본문 공백 제외: 일련번호 {id}번은 목록에만 존재하고 상세 본문이 없어 수집에서 제외합니다.")
                    continue
                    
                # 3. 검증을 통과한 데이터만 리스트에 추가
                prec_id_list_final.append(id)

            except Exception as e:
                print(f"처리 중 에러 발생(일련번호 {id}): {e}")
                continue
        # --- 반복문 종료 ---
        return prec_id_list_final

    except requests.exceptions.RequestException as e:
        print("요청 중 에러 발생:", e)
        return None

In [24]:
# 약관 판례 일련번호 저장
import pprint

result = search_prec_save("약관")

pprint.pprint(result)

⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 605903
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 605905
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 333690
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 303712
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 241496
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 213038
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 333804
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 209786
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 340520
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 341008
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 282742
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 292060
['605061',
 '605033',
 '242129',
 '235969',
 '221755',
 '174951',
 '149858',
 '149011',
 '143913',
 '146880',
 '168177',
 '70274',
 '84087',
 '115564',
 '197093',
 '602355',
 '189609',
 '72896']


In [38]:
import json

final_output_list = []

for i in result:
    response = search_prec_all(i)

    사건명 = response['PrecService'].get('사건명')
    사건번호 = response['PrecService'].get('사건번호')
    법원명 = response['PrecService'].get('법원명')
    사건종류명 = response['PrecService'].get('사건종류명')
    참조조문 = response['PrecService'].get('참조조문')
    참조판례 = response['PrecService'].get('첨조판례')
    판시사항 = response['PrecService'].get('판시사항')
    판결요지 = response['PrecService'].get('판결요지')
    판례내용 = response['PrecService'].get('판례내용')

    output_data = {
        "metadata": {
            "사건명": 사건명,
            "사건번호": 사건번호,
            "법원명": 법원명,
            "사건종류명": 사건종류명,
            "참조조문": 참조조문,
            "참조판례": 참조판례,
        },
        "content": {
            "판시사항": 판시사항,
            "판결요지": 판결요지,
            "판례내용": 판례내용
        }
    }
    
    final_output_list.append(output_data)


with open("약관_판례.json", "w", encoding='utf-8') as f:
    json.dump(final_output_list, f, ensure_ascii=False, indent=4)

---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200


In [ ]:
#계약 판례 일련번호 저장
import pprint

result = search_prec_save("계약")

len(result)

⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 619171
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 619047
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 618637
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 618383
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 618421
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 617365
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 618043
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 619175
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 618647
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 616793
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 617655
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 614859
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 617063
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 617419
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 617143
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 614975
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 616197
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 617663
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 616823
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 613651
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 613837
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 613845
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 614757
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 613835
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 613395
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 615257
⚠️ 경고: 데이터 구

685

In [41]:
import json

final_output_list = []

for i in result:
    response = search_prec_all(i)

    사건명 = response['PrecService'].get('사건명')
    사건번호 = response['PrecService'].get('사건번호')
    법원명 = response['PrecService'].get('법원명')
    사건종류명 = response['PrecService'].get('사건종류명')
    참조조문 = response['PrecService'].get('참조조문')
    참조판례 = response['PrecService'].get('첨조판례')
    판시사항 = response['PrecService'].get('판시사항')
    판결요지 = response['PrecService'].get('판결요지')
    판례내용 = response['PrecService'].get('판례내용')

    output_data = {
        "metadata": {
            "사건명": 사건명,
            "사건번호": 사건번호,
            "법원명": 법원명,
            "사건종류명": 사건종류명,
            "참조조문": 참조조문,
            "참조판례": 참조판례,
        },
        "content": {
            "판시사항": 판시사항,
            "판결요지": 판결요지,
            "판례내용": 판례내용
        }
    }
    
    final_output_list.append(output_data)


with open("계약_판례.json", "w", encoding='utf-8') as f:
    json.dump(final_output_list, f, ensure_ascii=False, indent=4)

---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---

In [ ]:
#임대차 판례 일련번호 저장
import pprint

result = search_prec_save("임대차")

len(result)

⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 618893
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 617365
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 618919
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 615045
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 617687
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 617621
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 607289
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 606255
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 613899
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 499123
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 242519
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 417806
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 604853
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 418010
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 418364
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 418768
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 418476
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 417808
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 245979
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 600655
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 346230
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 350316
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 344588
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 348374
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 337450
⚠️ 경고: 데이터 구조 오류 또는 누락 - 일련번호: 337648
⚠️ 경고: 데이터 구

225

In [48]:
import json

final_output_list = []

for i in result:
    response = search_prec_all(i)

    사건명 = response['PrecService'].get('사건명')
    사건번호 = response['PrecService'].get('사건번호')
    법원명 = response['PrecService'].get('법원명')
    사건종류명 = response['PrecService'].get('사건종류명')
    참조조문 = response['PrecService'].get('참조조문')
    참조판례 = response['PrecService'].get('첨조판례')
    판시사항 = response['PrecService'].get('판시사항')
    판결요지 = response['PrecService'].get('판결요지')
    판례내용 = response['PrecService'].get('판례내용')

    output_data = {
        "metadata": {
            "사건명": 사건명,
            "사건번호": 사건번호,
            "법원명": 법원명,
            "사건종류명": 사건종류명,
            "참조조문": 참조조문,
            "참조판례": 참조판례,
        },
        "content": {
            "판시사항": 판시사항,
            "판결요지": 판결요지,
            "판례내용": 판례내용
        }
    }
    
    final_output_list.append(output_data)


with open("임대차_판례.json", "w", encoding='utf-8') as f:
    json.dump(final_output_list, f, ensure_ascii=False, indent=4)

---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---
status = 200
---검색 결과 ---

# 법률 검색 후 json 저장하기

In [3]:
# 목록 조회
import os
import requests
import json
import pandas as pd

def search_law(query_text: str):
    """
    국가법령정보 API를 통해 법령을 검색하는 함수
    """
    
    # 기본 URL 설정
    base_url = "http://www.law.go.kr/DRF/lawSearch.do"


    params = {
        "OC": "legal_123",
        "target": "eflaw",
        "type": "json",
        "search":"1",
        "display": "3000",
        "query": query_text  # 여기에 검색어가 들어갑니다.
    }
    
    try:
        response = requests.get(base_url, params=params, timeout=30)
        
        print(f"--- '{query_text}' 검색 결과 ---")
        print("status =", response.status_code)
        
        # 성공적으로 데이터를 가져왔다면 JSON 형태로 출력하거나 반환
        if response.status_code == 200:
            return response.json()
        else:
            print("오류 발생:", response.text)
            return None
            
    except requests.exceptions.RequestException as e:
        print("요청 중 에러 발생:", e)
        return None

# # --- 사용 예시 ---
if __name__ == "__main__":
    result = search_law("약관의 규제에 관한 법률")

    law_list = result['LawSearch']['law']

    for law in law_list:
        print(f"ID: {law['id']} | 법령명: {law['법령약칭명']} | 시행일자: {law['시행일자']} | 법령ID : {law['법령ID']}")


#     # # 결과 출력
#     # if result:
#     #     import pprint
#     #     pprint.pprint(result)

--- '약관의 규제에 관한 법률' 검색 결과 ---
status = 200
ID: 1 | 법령명: 약관법 | 시행일자: 20240807 | 법령ID : 000667
ID: 2 | 법령명: 약관법 | 시행일자: 20240209 | 법령ID : 000667
ID: 3 | 법령명: 약관법 | 시행일자: 20240209 | 법령ID : 000667
ID: 4 | 법령명: 약관법 | 시행일자: 20231221 | 법령ID : 000667
ID: 5 | 법령명: 약관법 | 시행일자: 20211230 | 법령ID : 000667
ID: 6 | 법령명: 약관법 | 시행일자: 20181213 | 법령ID : 000667
ID: 7 | 법령명: 약관법 | 시행일자: 20180612 | 법령ID : 000667
ID: 8 | 법령명: 약관법 | 시행일자: 20160329 | 법령ID : 000667
ID: 9 | 법령명: 약관법 | 시행일자: 20130528 | 법령ID : 000667
ID: 10 | 법령명: 약관법 | 시행일자: 20120818 | 법령ID : 000667
ID: 11 | 법령명: 약관법 | 시행일자: 20110630 | 법령ID : 000667
ID: 12 | 법령명: 약관법 | 시행일자: 20101118 | 법령ID : 000667
ID: 13 | 법령명: 약관법 | 시행일자: 20100322 | 법령ID : 000667
ID: 14 | 법령명: 약관법 | 시행일자: 20080229 | 법령ID : 000667
ID: 15 | 법령명: 약관법 | 시행일자: 20070803 | 법령ID : 000667
ID: 16 | 법령명: 약관법 | 시행일자: 20070328 | 법령ID : 000667
ID: 17 | 법령명: 약관법 | 시행일자: 20050331 | 법령ID : 000667
ID: 18 | 법령명: 약관법 | 시행일자: 20040120 | 법령ID : 000667
ID: 19 | 법령명: 약관법 | 시행일자: 20010328 | 법령ID : 0006

In [4]:
result = search_law("하도급")

law_list = result['LawSearch']['law']

for law in law_list:
    print(f"ID: {law['id']} | 법령명: {law['법령약칭명']} | 시행일자: {law['시행일자']} | 법령ID : {law['법령ID']}")

--- '하도급' 검색 결과 ---
status = 200
ID: 1 | 법령명: 하도급법 | 시행일자: 20260811 | 법령ID : 001590
ID: 2 | 법령명: 하도급법 | 시행일자: 20251217 | 법령ID : 001590
ID: 3 | 법령명: 하도급법 | 시행일자: 20251002 | 법령ID : 001590
ID: 4 | 법령명: 하도급법 | 시행일자: 20250401 | 법령ID : 001590
ID: 5 | 법령명: 하도급법 | 시행일자: 20250121 | 법령ID : 001590
ID: 6 | 법령명: 하도급법 | 시행일자: 20240828 | 법령ID : 001590
ID: 7 | 법령명: 하도급법 | 시행일자: 20240807 | 법령ID : 001590
ID: 8 | 법령명: 하도급법 | 시행일자: 20240209 | 법령ID : 001590
ID: 9 | 법령명: 하도급법 | 시행일자: 20240209 | 법령ID : 001590
ID: 10 | 법령명: 하도급법 | 시행일자: 20231004 | 법령ID : 001590
ID: 11 | 법령명: 하도급법 | 시행일자: 20230718 | 법령ID : 001590
ID: 12 | 법령명: 하도급법 | 시행일자: 20230112 | 법령ID : 001590
ID: 13 | 법령명: 하도급법 | 시행일자: 20220712 | 법령ID : 001590
ID: 14 | 법령명: 하도급법 | 시행일자: 20220218 | 법령ID : 001590
ID: 15 | 법령명: 하도급법 | 시행일자: 20211230 | 법령ID : 001590
ID: 16 | 법령명: 하도급법 | 시행일자: 20210817 | 법령ID : 001590
ID: 17 | 법령명: 하도급법 | 시행일자: 20201210 | 법령ID : 001590
ID: 18 | 법령명: 하도급법 | 시행일자: 20201210 | 법령ID : 001590
ID: 19 | 법령명: 하도급법 | 시행일자: 20200527 | 법령

In [5]:
result = search_law("독점규제 및 공정거래에 관한 법률")


law_list = result['LawSearch']['law']

for law in law_list:
    print(f"ID: {law['id']} | 법령명: {law['법령약칭명']} | 시행일자: {law['시행일자']} | 법령ID : {law['법령ID']}")

--- '독점규제 및 공정거래에 관한 법률' 검색 결과 ---
status = 200
ID: 1 | 법령명: 공정거래법 | 시행일자: 20270207 | 법령ID : 001591
ID: 2 | 법령명: 공정거래법 | 시행일자: 20260512 | 법령ID : 001591
ID: 3 | 법령명: 공정거래법 | 시행일자: 20251001 | 법령ID : 001591
ID: 4 | 법령명: 공정거래법 | 시행일자: 20251001 | 법령ID : 001591
ID: 5 | 법령명: 공정거래법 | 시행일자: 20250422 | 법령ID : 001591
ID: 6 | 법령명: 공정거래법 | 시행일자: 20250121 | 법령ID : 001591
ID: 7 | 법령명: 공정거래법 | 시행일자: 20240807 | 법령ID : 001591
ID: 8 | 법령명: 공정거래법 | 시행일자: 20240710 | 법령ID : 001591
ID: 9 | 법령명: 공정거래법 | 시행일자: 20240621 | 법령ID : 001591
ID: 10 | 법령명: 공정거래법 | 시행일자: 20240209 | 법령ID : 001591
ID: 11 | 법령명: 공정거래법 | 시행일자: 20240209 | 법령ID : 001591
ID: 12 | 법령명: 공정거래법 | 시행일자: 20240206 | 법령ID : 001591
ID: 13 | 법령명: 공정거래법 | 시행일자: 20231221 | 법령ID : 001591
ID: 14 | 법령명: 공정거래법 | 시행일자: 20231221 | 법령ID : 001591
ID: 15 | 법령명: 공정거래법 | 시행일자: 20230620 | 법령ID : 001591
ID: 16 | 법령명: 공정거래법 | 시행일자: 20221230 | 법령ID : 001591
ID: 17 | 법령명: 공정거래법 | 시행일자: 20220629 | 법령ID : 001591
ID: 18 | 법령명: 공정거래법 | 시행일자: 20211230 | 법령ID : 001591
ID: 19 

In [6]:
result = search_law("민법")


law_list = result['LawSearch']['law']

for law in law_list:
    print(f"ID: {law['id']} | 법령명: {law['법령명한글']} | 시행일자: {law['시행일자']} | 법령ID : {law['법령ID']}")

--- '민법' 검색 결과 ---
status = 200
ID: 1 | 법령명: 난민법 | 시행일자: 20161220 | 법령ID : 011546
ID: 2 | 법령명: 난민법 | 시행일자: 20140619 | 법령ID : 011546
ID: 3 | 법령명: 난민법 | 시행일자: 20130701 | 법령ID : 011546
ID: 4 | 법령명: 난민법 시행규칙 | 시행일자: 20241203 | 법령ID : 011885
ID: 5 | 법령명: 난민법 시행규칙 | 시행일자: 20231227 | 법령ID : 011885
ID: 6 | 법령명: 난민법 시행규칙 | 시행일자: 20230227 | 법령ID : 011885
ID: 7 | 법령명: 난민법 시행규칙 | 시행일자: 20220101 | 법령ID : 011885
ID: 8 | 법령명: 난민법 시행규칙 | 시행일자: 20210729 | 법령ID : 011885
ID: 9 | 법령명: 난민법 시행규칙 | 시행일자: 20191231 | 법령ID : 011885
ID: 10 | 법령명: 난민법 시행규칙 | 시행일자: 20180515 | 법령ID : 011885
ID: 11 | 법령명: 난민법 시행규칙 | 시행일자: 20130701 | 법령ID : 011885
ID: 12 | 법령명: 난민법 시행령 | 시행일자: 20250919 | 법령ID : 011878
ID: 13 | 법령명: 난민법 시행령 | 시행일자: 20241203 | 법령ID : 011878
ID: 14 | 법령명: 난민법 시행령 | 시행일자: 20220712 | 법령ID : 011878
ID: 15 | 법령명: 난민법 시행령 | 시행일자: 20220218 | 법령ID : 011878
ID: 16 | 법령명: 난민법 시행령 | 시행일자: 20210727 | 법령ID : 011878
ID: 17 | 법령명: 난민법 시행령 | 시행일자: 20201215 | 법령ID : 011878
ID: 18 | 법령명: 난민법 시행령 | 시행일자: 20191231 | 법령ID 

In [7]:
result = search_law("민법")
law_list = result['LawSearch']['law']

for law in law_list:
    # 법령명이 정확히 '민법'과 일치하는 경우만 출력
    if law['법령명한글'] == '민법':
        print(f"ID: {law['id']} | 법령명: {law['법령명한글']} | 시행일자: {law['시행일자']} | 법령ID : {law['법령ID']}")

--- '민법' 검색 결과 ---
status = 200
ID: 22 | 법령명: 민법 | 시행일자: 20260317 | 법령ID : 001706
ID: 23 | 법령명: 민법 | 시행일자: 20260101 | 법령ID : 001706
ID: 24 | 법령명: 민법 | 시행일자: 20250131 | 법령ID : 001706
ID: 25 | 법령명: 민법 | 시행일자: 20240517 | 법령ID : 001706
ID: 26 | 법령명: 민법 | 시행일자: 20230628 | 법령ID : 001706
ID: 27 | 법령명: 민법 | 시행일자: 20221213 | 법령ID : 001706
ID: 28 | 법령명: 민법 | 시행일자: 20210126 | 법령ID : 001706
ID: 29 | 법령명: 민법 | 시행일자: 20201020 | 법령ID : 001706
ID: 30 | 법령명: 민법 | 시행일자: 20180201 | 법령ID : 001706
ID: 31 | 법령명: 민법 | 시행일자: 20170603 | 법령ID : 001706
ID: 32 | 법령명: 민법 | 시행일자: 20161220 | 법령ID : 001706
ID: 33 | 법령명: 민법 | 시행일자: 20160204 | 법령ID : 001706
ID: 34 | 법령명: 민법 | 시행일자: 20160106 | 법령ID : 001706
ID: 35 | 법령명: 민법 | 시행일자: 20151016 | 법령ID : 001706
ID: 36 | 법령명: 민법 | 시행일자: 20150701 | 법령ID : 001706
ID: 37 | 법령명: 민법 | 시행일자: 20141230 | 법령ID : 001706
ID: 38 | 법령명: 민법 | 시행일자: 20130701 | 법령ID : 001706
ID: 39 | 법령명: 민법 | 시행일자: 20130701 | 법령ID : 001706
ID: 40 | 법령명: 민법 | 시행일자: 20130701 | 법령ID : 001706
ID: 41 | 법령명: 민법 |

### 본문 조회

In [8]:
# 본문 조회
from rich import print as rprint
import os
import requests
import json
import pandas as pd

def search_law(code:str):
    """
    국가법령정보 API를 통해 법령을 검색하는 함수
    """
    
    # 기본 URL 설정
    base_url = "http://www.law.go.kr/DRF/lawService.do"


    params = {
        "OC": "legal_123",
        "target": "eflaw",
        "type": "json",
        "display": "1000",
        "ID" : code
    }
    
    try:
        response = requests.get(base_url, params=params, timeout=30)
        
        print(f"--- 검색 결과 ---")
        print("status =", response.status_code)
        
        # 성공적으로 데이터를 가져왔다면 JSON 형태로 출력하거나 반환
        if response.status_code == 200:
            return response.json()
        else:
            print("오류 발생:", response.text)
            return None
            
    except requests.exceptions.RequestException as e:
        print("요청 중 에러 발생:", e)
        return None


            

In [9]:
# 약관법
result = search_law("000667")

# 결과 출력
if result:
    import pprint
    pprint.pprint(result)


--- 검색 결과 ---
status = 200
{'법령': {'개정문': {'개정문내용': [['⊙법률 제20239호(2024.2.6)',
                           '독점규제 및 공정거래에 관한 법률 일부개정법률',
                           '[본문 생략]',
                           '        부칙',
                           '제1조(시행일) 이 법은 공포 후 6개월이 경과한 날부터 시행한다. <단서 생략>',
                           '제2조부터 제5조까지 생략',
                           '제6조(다른 법률의 개정) ①부터 ④까지 생략',
                           '  ⑤ 약관의 규제에 관한 법률 일부를 다음과 같이 개정한다.',
                           '  제30조의2제2항 중 "「독점규제 및 공정거래에 관한 법률」 제96조부터 '
                           '제101조까지의 규정을"을 "「독점규제 및 공정거래에 관한 법률」 제96조부터 '
                           '제98조까지, 제98조의2, 제98조의3 및 제99조부터 제101조까지를"로 한다.',
                           '  ⑥부터 ⑨까지 생략']]},
        '기본정보': {'공동부령정보': '',
                 '공포번호': '20239',
                 '공포법령여부': 'N',
                 '공포일자': '20240206',
                 '법령ID': '000667',
                 '법령명_한글': '약관의 규제에 관한 법률',
                 '법령명_한자': '약관의 규제에 관한 법률',
                 

In [10]:
import json

def parse_law_data(raw_data):
    # 내장 dict, list 키워드가 오염되었을 경우를 대비해 안전한 타입 객체 확보
    _dict_type = type({})
    _list_type = type([])
    _str_type = type("")

    # 1. 기본 정보 추출
    base_info = raw_data.get('법령', {}).get('기본정보', {})
    law_name = base_info.get('법령명_한글', '')
    law_id = base_info.get('법령ID', '')
    
    dept_data = base_info.get('소관부처', '')
    if isinstance(dept_data, _dict_type):
        dept = dept_data.get('content', '')
    else:
        dept = dept_data
        
    enforce_date = base_info.get('시행일자', '')

    articles_list = raw_data.get('법령', {}).get('조문', {}).get('조문단위', [])
    result_articles = []

    # 텍스트가 문자열인지 리스트인지 판별하여 안전하게 문자열로 리턴하는 내부 헬퍼 함수
    def clean_text(val):
        if not val:
            return ""
        if isinstance(val, _str_type):
            return val.strip()
        if isinstance(val, _list_type):
            # 리스트 내부 원소들을 공백으로 엮어서 하나의 문자열로 합침
            return " ".join([str(v).strip() for v in val if v])
        return str(val).strip()

    # 항, 호, 목 하위 구조를 안전하게 순회하며 텍스트를 추출하는 보조 함수
    def collect_texts(node):
        texts = []
        if node is None or not node:
            return texts
        
        nodes = node if isinstance(node, _list_type) else [node]
        
        for n in nodes:
            if n is None or not isinstance(n, _dict_type):
                continue
                
            # 현재 계층의 텍스트 추출 (리스트 예외 방어 적용)
            for key in ['항내용', '호내용', '목내용']:
                if n.get(key):
                    cleaned = clean_text(n[key])
                    if cleaned:
                        texts.append(cleaned)
            
            # 하위 계층(호, 목)이 None이 아닐 때만 재귀 호출
            if n.get('호') is not None:
                texts.extend(collect_texts(n['호']))
            if n.get('목') is not None:
                texts.extend(collect_texts(n['목']))
                
        return texts

    # 2. 조문 단위 순회 및 내용 병합
    for item in articles_list:
        if item.get('조문여부') != '조문':
            continue

        # --- 조문 번호 결합 로직 수정 ---
        article_no = str(item.get('조문번호', '')).strip()
        sub_no = str(item.get('조문가지번호', '')).strip()
        
        # 가지번호가 존재하면 '번호_가지번호' 형태로 결합 (예: 17_2)
        if sub_no and sub_no != '0' and sub_no != 'None':
            article_no = f"{article_no}_{sub_no}"
        # ---------------------------------

        article_title = item.get('조문제목', '')
        
        content_parts = []
        if item.get('조문내용'):
            # 여기도 리스트 예외 방어 적용
            cleaned_main = clean_text(item['조문내용'])
            if cleaned_main:
                content_parts.append(cleaned_main)

        # 하위 구조 탐색
        if '항' in item:
            content_parts.extend(collect_texts(item['항']))
        elif '호' in item:
            content_parts.extend(collect_texts(item['호']))

        # 줄바꿈으로 본문 및 하위 항목 병합
        merged_content = "\n".join([p for p in content_parts if p])

        result_articles.append({
            "조문번호": article_no,
            "조문제목": article_title,
            "조문내용": merged_content
        })

    # 3. 최종 결과 구조화
    output_data = {
        "법령명한글": law_name,
        "법령ID": law_id,
        "소관부처": dept,
        "시행일자": enforce_date,
        "조문리스트": result_articles
    }
    
    return output_data

In [11]:
약관법 = search_law("000667")

# 첫 번째 데이터 변환 및 JSON 출력
parsed_data1 = parse_law_data(약관법)
json_output1 = json.dumps(parsed_data1, ensure_ascii=False, indent=2)
print("--- 약관법 정제 결과 ---")
print(json_output1)


file_path = "약관법.json"
with open(file_path, "w", encoding="utf-8") as f:
    json.dump(parsed_data1, f, ensure_ascii=False, indent=4)

--- 검색 결과 ---
status = 200
--- 약관법 정제 결과 ---
{
  "법령명한글": "약관의 규제에 관한 법률",
  "법령ID": "000667",
  "소관부처": "공정거래위원회",
  "시행일자": "20240807",
  "조문리스트": [
    {
      "조문번호": "1",
      "조문제목": "목적",
      "조문내용": "제1조(목적) 이 법은 사업자가 그 거래상의 지위를 남용하여 불공정한 내용의 약관(約款)을 작성하여 거래에 사용하는 것을 방지하고 불공정한 내용의 약관을 규제함으로써 건전한 거래질서를 확립하고, 이를 통하여 소비자를 보호하고 국민생활을 균형 있게 향상시키는 것을 목적으로 한다."
    },
    {
      "조문번호": "2",
      "조문제목": "정의",
      "조문내용": "제2조(정의) 이 법에서 사용하는 용어의 정의는 다음과 같다.\n1.  \"약관\"이란 그 명칭이나 형태 또는 범위에 상관없이 계약의 한쪽 당사자가 여러 명의 상대방과 계약을 체결하기 위하여 일정한 형식으로 미리 마련한 계약의 내용을 말한다.\n2.  \"사업자\"란 계약의 한쪽 당사자로서 상대 당사자에게 약관을 계약의 내용으로 할 것을 제안하는 자를 말한다.\n3.  \"고객\"이란 계약의 한쪽 당사자로서 사업자로부터 약관을 계약의 내용으로 할 것을 제안받은 자를 말한다."
    },
    {
      "조문번호": "3",
      "조문제목": "약관의 작성 및 설명의무 등",
      "조문내용": "제3조(약관의 작성 및 설명의무 등)\n① 사업자는 고객이 약관의 내용을 쉽게 알 수 있도록 한글로 작성하고, 표준화ㆍ체계화된 용어를 사용하며, 약관의 중요한 내용을 부호, 색채, 굵고 큰 문자 등으로 명확하게 표시하여 알아보기 쉽게 약관을 작성하여야 한다. <개정 2011.3.29>\n② 사업자는 계약을 체결할 때에는 고객에게 약관의 내용을 계약의 종류에 따라 일반적으로 예상되는 방

In [12]:
약관법_시행령 = search_law("004135")

# 두 번째 데이터 변환 및 JSON 출력
parsed_data2 = parse_law_data(약관법_시행령)
json_output2 = json.dumps(parsed_data2, ensure_ascii=False, indent=2)
print("\n--- 약관법 시행령 정제 결과 ---")
print(json_output2)



file_path = "약관법_시행령.json"
with open(file_path, "w", encoding="utf-8") as f:
    json.dump(parsed_data2, f, ensure_ascii=False, indent=4)


--- 검색 결과 ---
status = 200

--- 약관법 시행령 정제 결과 ---
{
  "법령명한글": "약관의 규제에 관한 법률 시행령",
  "법령ID": "004135",
  "소관부처": "공정거래위원회",
  "시행일자": "20231221",
  "조문리스트": [
    {
      "조문번호": "1",
      "조문제목": "목적",
      "조문내용": "제1조(목적) 이 영은 「약관의 규제에 관한 법률」에서 위임된 사항과 그 시행에 필요한 사항을 규정함을 목적으로 한다."
    },
    {
      "조문번호": "2",
      "조문제목": "약관의 비치",
      "조문내용": "제2조(약관의 비치) 「약관의 규제에 관한 법률」(이하 \"법\"이라 한다) 제3조제2항 각 호에 해당하는 업종의 약관인 경우에도 사업자는 영업소에 해당 약관을 비치하여 고객이 볼 수 있도록 하여야 한다."
    },
    {
      "조문번호": "3",
      "조문제목": "적용의 제한",
      "조문내용": "제3조(적용의 제한) 법 제15조에 따라 다음 각 호의 어느 하나에 해당하는 업종의 약관에 대해서는 법 제7조부터 제14조까지의 규정을 적용하지 아니한다.\n1.  국제적으로 통용되는 운송업\n2.  국제적으로 통용되는 금융업 및 보험업\n3.  「무역보험법」에 따른 무역보험"
    },
    {
      "조문번호": "4",
      "조문제목": "시정 조치의 방식",
      "조문내용": "제4조(시정 조치의 방식) 법 제17조의2에 따른 시정권고 또는 시정명령은 그 내용을 분명히 밝힌 서면으로 하여야 한다."
    },
    {
      "조문번호": "5",
      "조문제목": "시정 조치의 요청 및 권고",
      "조문내용": "제5조(시정 조치의 요청 및 권고)\n① 법 제18조에 따른 시정에 필요한 조치의 요청 또는 권고는 그 내용을 분명히 밝힌 서면(전자문서

In [13]:
하도급법 = search_law("001590")


parsed_data = parse_law_data(하도급법)
json_output = json.dumps(parsed_data, ensure_ascii=False, indent=2)
print("\n--- 정제 결과 ---")
print(json_output)



file_path = "하도급법.json"
with open(file_path, "w", encoding="utf-8") as f:
    json.dump(parsed_data, f, ensure_ascii=False, indent=4)


--- 검색 결과 ---
status = 200

--- 정제 결과 ---
{
  "법령명한글": "하도급거래 공정화에 관한 법률",
  "법령ID": "001590",
  "소관부처": "공정거래위원회",
  "시행일자": "20251217",
  "조문리스트": [
    {
      "조문번호": "1",
      "조문제목": "목적",
      "조문내용": "제1조(목적) 이 법은 공정한 하도급거래질서를 확립하여 원사업자(原事業者)와 수급사업자(受給事業者)가 대등한 지위에서 상호보완하며 균형 있게 발전할 수 있도록 함으로써 국민경제의 건전한 발전에 이바지함을 목적으로 한다."
    },
    {
      "조문번호": "2",
      "조문제목": "정의",
      "조문내용": "제2조(정의)\n① 이 법에서 \"하도급거래\"란 원사업자가 수급사업자에게 제조위탁(가공위탁을 포함한다. 이하 같다)ㆍ수리위탁ㆍ건설위탁 또는 용역위탁을 하거나 원사업자가 다른 사업자로부터 제조위탁ㆍ수리위탁ㆍ건설위탁 또는 용역위탁을 받은 것을 수급사업자에게 다시 위탁한 경우, 그 위탁(이하 \"제조등의 위탁\"이라 한다)을 받은 수급사업자가 위탁받은 것(이하 \"목적물등\"이라 한다)을 제조ㆍ수리ㆍ시공하거나 용역수행하여 원사업자에게 납품ㆍ인도 또는 제공(이하 \"납품등\"이라 한다)하고 그 대가(이하 \"하도급대금\"이라 한다)를 받는 행위를 말한다.\n② 이 법에서 \"원사업자\"란 다음 각 호의 어느 하나에 해당하는 자를 말한다. <개정 2011.3.29, 2014.5.28, 2015.7.24>\n1.  중소기업자(「중소기업기본법」 제2조제1항 또는 제3항에 따른 자를 말하며, 「중소기업협동조합법」에 따른 중소기업협동조합을 포함한다. 이하 같다)가 아닌 사업자로서 중소기업자에게 제조등의 위탁을 한 자\n2.  중소기업자 중 직전 사업연도의 연간매출액[관계 법률에 따라 시공능력평가액을 적용받는 거래의 경우에는 하도급계약 체결 당시 공시된 시공능력평가액의 

In [14]:
하도급법_시행령 = search_law("005367")


parsed_data = parse_law_data(하도급법_시행령)
json_output = json.dumps(parsed_data, ensure_ascii=False, indent=2)
print("\n--- 정제 결과 ---")
print(json_output)



file_path = "하도급법_시행령.json"
with open(file_path, "w", encoding="utf-8") as f:
    json.dump(parsed_data, f, ensure_ascii=False, indent=4)

--- 검색 결과 ---
status = 200

--- 정제 결과 ---
{
  "법령명한글": "하도급거래 공정화에 관한 법률 시행령",
  "법령ID": "005367",
  "소관부처": "공정거래위원회",
  "시행일자": "20251001",
  "조문리스트": [
    {
      "조문번호": "1",
      "조문제목": "목적",
      "조문내용": "제1조(목적) 이 영은 「하도급거래 공정화에 관한 법률」에서 위임한 사항과 그 시행에 필요한 사항을 규정함을 목적으로 한다."
    },
    {
      "조문번호": "2",
      "조문제목": "중소기업자의 범위 등",
      "조문내용": "제2조(중소기업자의 범위 등)\n① 「하도급거래 공정화에 관한 법률」(이하 \"법\"이라 한다) 제2조제2항제2호 본문에 따른 연간매출액은 하도급계약을 체결하는 사업연도의 직전 사업연도의 손익계산서에 표시된 매출액으로 한다. 다만, 직전 사업연도 중에 사업을 시작한 경우에는 직전 사업연도의 매출액을 1년으로 환산한 금액으로 하며, 해당 사업연도에 사업을 시작한 경우에는 사업 시작일부터 하도급계약 체결일까지의 매출액을 1년으로 환산한 금액으로 한다.\n② 법 제2조제2항제2호 본문에 따른 자산총액은 하도급계약을 체결하는 사업연도의 직전 사업연도 종료일 현재의 재무상태표에 표시된 자산총액으로 한다. 다만, 해당 사업연도에 사업을 시작한 경우에는 사업 시작일 현재의 재무상태표에 표시된 자산총액으로 한다. <개정 2021.1.5, 2020.1.12>\n③ 삭제 <2016.1.22>\n④ 법 제2조제2항제2호 단서에서 \"대통령령으로 정하는 연간매출액에 해당하는 중소기업자\"란 다음 각 호에 해당하는 자를 말한다. <개정 2021.1.12>\n1.  제조위탁ㆍ수리위탁의 경우: 연간매출액이 30억원 미만인 중소기업자\n2.  건설위탁의 경우: 시공능력평가액이 45억원 미만인 중소기업자\n3.  용역위탁의 경우: 연간매출액이 10억원 미

In [15]:
공정거래법 = search_law("001591")


parsed_data = parse_law_data(공정거래법)
json_output = json.dumps(parsed_data, ensure_ascii=False, indent=2)
print("\n--- 정제 결과 ---")
print(json_output)



file_path = "공정거래법.json"
with open(file_path, "w", encoding="utf-8") as f:
    json.dump(parsed_data, f, ensure_ascii=False, indent=4)

--- 검색 결과 ---
status = 200

--- 정제 결과 ---
{
  "법령명한글": "독점규제 및 공정거래에 관한 법률",
  "법령ID": "001591",
  "소관부처": "공정거래위원회",
  "시행일자": "20260512",
  "조문리스트": [
    {
      "조문번호": "1",
      "조문제목": "목적",
      "조문내용": "제1조(목적) 이 법은 사업자의 시장지배적지위의 남용과 과도한 경제력의 집중을 방지하고, 부당한 공동행위 및 불공정거래행위를 규제하여 공정하고 자유로운 경쟁을 촉진함으로써 창의적인 기업활동을 조성하고 소비자를 보호함과 아울러 국민경제의 균형 있는 발전을 도모함을 목적으로 한다."
    },
    {
      "조문번호": "2",
      "조문제목": "정의",
      "조문내용": "제2조(정의) 이 법에서 사용하는 용어의 뜻은 다음과 같다. <개정 2025.10.1>\n1.  \"사업자\"란 제조업, 서비스업 또는 그 밖의 사업을 하는 자를 말한다. 이 경우 사업자의 이익을 위한 행위를 하는 임원, 종업원(계속하여 회사의 업무에 종사하는 사람으로서 임원 외의 사람을 말한다. 이하 같다), 대리인 및 그 밖의 자는 사업자단체에 관한 규정을 적용할 때에는 사업자로 본다.\n2.  \"사업자단체\"란 그 형태가 무엇이든 상관없이 둘 이상의 사업자가 공동의 이익을 증진할 목적으로 조직한 결합체 또는 그 연합체를 말한다.\n3.  \"시장지배적사업자\"란 일정한 거래분야의 공급자나 수요자로서 단독으로 또는 다른 사업자와 함께 상품이나 용역의 가격, 수량, 품질, 그 밖의 거래조건을 결정ㆍ유지 또는 변경할 수 있는 시장지위를 가진 사업자를 말한다. 이 경우 시장지배적사업자를 판단할 때에는 시장점유율, 진입장벽의 존재 및 정도, 경쟁사업자의 상대적 규모 등을 종합적으로 고려한다.\n4.  \"일정한 거래분야\"란 거래의 객체별ㆍ단계별 또는 지역별로 경쟁관계에 있거나 경쟁관계가 성립될

In [16]:
공정거래법_시행령 = search_law("003440")


parsed_data = parse_law_data(공정거래법_시행령)
json_output = json.dumps(parsed_data, ensure_ascii=False, indent=2)
print("\n--- 정제 결과 ---")
print(json_output)



file_path = "공정거래법_시행령.json"
with open(file_path, "w", encoding="utf-8") as f:
    json.dump(parsed_data, f, ensure_ascii=False, indent=4)

--- 검색 결과 ---
status = 200

--- 정제 결과 ---
{
  "법령명한글": "독점규제 및 공정거래에 관한 법률 시행령",
  "법령ID": "003440",
  "소관부처": "공정거래위원회",
  "시행일자": "20260324",
  "조문리스트": [
    {
      "조문번호": "1",
      "조문제목": "목적",
      "조문내용": "제1조(목적) 이 영은 「독점규제 및 공정거래에 관한 법률」에서 위임된 사항과 그 시행에 필요한 사항을 규정함을 목적으로 한다."
    },
    {
      "조문번호": "2",
      "조문제목": "시장지배적사업자의 기준",
      "조문내용": "제2조(시장지배적사업자의 기준)\n① 「독점규제 및 공정거래에 관한 법률」(이하 \"법\"이라 한다) 제2조제3호 후단에 따른 시장점유율은 법 제5조를 위반한 혐의가 있는 행위의 종료일이 속하는 사업연도의 직전 1년 동안에 국내에서 공급되거나 구매된 상품이나 용역의 금액 중 해당 사업자가 같은 기간 동안 국내에서 공급하거나 구매한 상품이나 용역의 금액의 비율로 한다. 다만, 시장점유율을 금액기준으로 산정하기 어려운 경우에는 물량기준 또는 생산능력기준으로 산정할 수 있다.\n② 법 제2조제3호에 따라 시장지배적사업자를 판단하는 경우에는 해당 사업자와 그 계열회사를 하나의 사업자로 본다.\n③ 제1항 및 제2항에서 규정한 사항 외에 시장지배적사업자의 판단에 필요한 세부기준은 공정거래위원회가 정하여 고시한다."
    },
    {
      "조문번호": "3",
      "조문제목": "지주회사의 기준",
      "조문내용": "제3조(지주회사의 기준)\n① 법 제2조제7호 전단에서 \"자산총액이 대통령령으로 정하는 금액 이상인 회사\"란 다음 각 호의 구분에 따른 회사를 말한다.\n1.  해당 사업연도에 설립되었거나 합병 또는 분할ㆍ분할합병ㆍ물적분할(이하 \"분할\"이라 한다)을 한 경우: 설립등기일ㆍ합병등기